# The Transformer: A Principled Derivation from First Principles

**Or: How to Actually Understand What the Fuck is Going On**

---

## Preface: Why Most Explanations Are Garbage

Most transformer explanations start with "attention is like looking at things" and proceed to throw matrices at you without ever explaining *why* any of this works. This is backwards. We're going to start from the end — from what the transformer is actually trying to do — and derive everything else from that objective.

**The thesis:** A transformer is a maximum likelihood estimator that predicts categorical distributions over vocabulary tokens, and every single architectural choice exists to reduce entropy (uncertainty) in that prediction.

---

# Part 1: Starting from the End — The Objective Function

## 1.1 What is a Transformer Actually Doing?

A language model receives a sequence of tokens $x_1, x_2, \ldots, x_{t-1}$ and outputs a **probability distribution** over what the next token $x_t$ should be. That's it. Everything else is implementation details.

Formally, the model learns a function:

$$f_\theta: \mathcal{V}^{t-1} \rightarrow \Delta^{|\mathcal{V}|-1}$$

where:
- $\mathcal{V}$ is the vocabulary (set of all possible tokens)
- $\mathcal{V}^{t-1}$ is the space of all possible sequences of length $t-1$
- $\Delta^{|\mathcal{V}|-1}$ is the $(|\mathcal{V}|-1)$-dimensional probability simplex (just logits after softmax, constrained to sum to 1, all non-negative)
- $\theta$ are the learnable parameters

**The output is always a probability distribution over vocabulary.** Always. No matter what task you fine-tune for, the fundamental operation is predicting token probabilities.

## Concrete Example: From Input to Probability Distribution

**Setup:**
- Vocabulary $\mathcal{V} = \{\text{the}, \text{cat}, \text{sat}, \text{mat}\}$, indexed as $\{0, 1, 2, 3\}$
- Vocabulary size $|\mathcal{V}| = 4$
- Input sequence: "the cat sat" (tokens $[0, 1, 2]$)
- We predict position $t = 4$

**The function signature:**

$$f_\theta: \mathcal{V}^{t-1} \rightarrow \Delta^{|\mathcal{V}|-1}$$

The domain $\mathcal{V}^{t-1} = \mathcal{V}^3$ contains all $4^3 = 64$ possible 3-token sequences. Our input "the cat sat" is one specific element of this set.

**Step 1: Forward pass produces logits**

The transformer processes the input and outputs raw scores (logits) $\mathbf{z} \in \mathbb{R}^4$:

$$\mathbf{z} = f_\theta^{\text{logits}}(\text{the, cat, sat}) = \begin{pmatrix} -1.2 \\ 0.5 \\ -0.3 \\ 2.1 \end{pmatrix}$$

These are unconstrained — any real values.

**Step 2: Softmax maps logits to the simplex**

$$\hat{p}_k = \frac{e^{z_k}}{\sum_{j=0}^{3} e^{z_j}}$$

Computing each term:

| Token | $k$ | $z_k$ | $e^{z_k}$ |
|-------|-----|-------|-----------|
| the   | 0   | -1.2  | 0.301     |
| cat   | 1   |  0.5  | 1.649     |
| sat   | 2   | -0.3  | 0.741     |
| mat   | 3   |  2.1  | 8.166     |

Sum: $\sum_j e^{z_j} = 0.301 + 1.649 + 0.741 + 8.166 = 10.857$

Final probabilities:

$$\hat{\mathbf{p}} = \text{softmax}(\mathbf{z}) = \begin{pmatrix} 0.028 \\ 0.152 \\ 0.068 \\ 0.752 \end{pmatrix} \in \Delta^{|\mathcal{V}|-1} = \Delta^{4-1}$$

**Interpretation:** Given "the cat sat", the model predicts:
- P(the) = 2.8%
- P(cat) = 15.2%  
- P(sat) = 6.8%
- P(mat) = 75.2% ← most likely next token

**Verification:** $0.028 + 0.152 + 0.068 + 0.752 = 1.0$ ✓

The output lies on the $(|\mathcal{V}|-1)$-simplex: four non-negative values summing to 1, meaning 3 degrees of freedom.

## TODO: create a widget like here to explain softmax: [Softmax Function Explained In Depth with 3D Visuals](https://youtu.be/ytbYRIN0N4g?si=mmknveJJ0EzdTK-Y)

Add notes on temperature!

## 1.2 The Categorical Distribution: What We're Predicting

Let's be precise about what kind of distribution we're dealing with.

**Definition (Categorical Distribution):** A categorical distribution is a discrete probability distribution over $K$ mutually exclusive categories. If $X$ is a random variable taking values in $\{1, 2, \ldots, K\}$, then $X$ follows a categorical distribution with parameters $\pi = (\pi_1, \pi_2, \ldots, \pi_K)$ if:

$$P(X = k) = \pi_k \quad \text{for } k \in \{1, 2, \ldots, K\}$$

with the constraints:
1. $\pi_k \geq 0$ for all $k$ (non-negativity)
2. $\sum_{k=1}^{K} \pi_k = 1$ (normalization)

**Why categorical?** Because the next token must be exactly one token from the vocabulary. It can't be half "cat" and half "dog". It's a discrete choice among $|\mathcal{V}|$ mutually exclusive options.

**The probability mass function (PMF):** For a single observation $x \in \{1, \ldots, K\}$:

$$P(X = x | \pi) = \prod_{k=1}^{K} \pi_k^{\mathbb{1}[x = k]}$$

where $\mathbb{1}[x = k]$ is the indicator function that equals 1 if $x = k$ and 0 otherwise.

**Why this weird product form?** Let's verify it's correct. If $x = 3$, then $\mathbb{1}[x=k] = 0$ for all $k \neq 3$, and $\mathbb{1}[x=3] = 1$. So:

$$P(X = 3 | \pi) = \pi_1^0 \cdot \pi_2^0 \cdot \pi_3^1 \cdot \pi_4^0 \cdots = 1 \cdot 1 \cdot \pi_3 \cdot 1 \cdots = \pi_3$$

Good. This form will be essential for deriving the loss function.

> **Note on the Product Notation:** The formula $P(X = x | \pi) = \prod_{k=1}^{K} \pi_k^{\mathbb{1}[x = k]}$ is a mathematical convenience for derivations, not how you compute anything in practice. In code, you simply index into the probability vector: `loss = -log(probs[target])`. The indicator function $\mathbb{1}[x = k]$ acts as a "selector" — it equals 1 only for the true token (from training data), zeroing out all other terms. This lets us write a single formula that works for any ground truth class, which is essential when deriving the cross-entropy loss algebraically. Do not confuse $x$ (the ground truth label we're training on) with argmax (what the model would predict at inference).

> **Note on MLE and Parameters:** The learnable parameters are the network weights $\theta$, not the probabilities $\pi$. The probability vector $\pi = \text{softmax}(f_\theta(x_{<t}))$ is an intermediate output computed from $\theta$. When we say "maximize likelihood," we mean: find weights $\theta^*$ such that the probability assigned to the true token is as high as possible. If the true token is "mat" (index 3), we want $\pi_3$ to be large. MLE adjusts $\theta$ until the network outputs distributions that put most of their mass on whatever token actually comes next in the training data. The loss $-\log \pi_x$ is small when $\pi_x$ is close to 1 (model confident and correct) and large when $\pi_x$ is close to 0 (model assigned little probability to the truth). Gradient descent on this loss pushes $\theta$ toward configurations where the correct $\pi$ is maximized.

## 1.3 Maximum Likelihood Estimation: The Training Objective

**The Setup:** We have a dataset $\mathcal{D} = \{(x_1^{(i)}, x_2^{(i)}, \ldots, x_{T_i}^{(i)})\}_{i=1}^N$ of $N$ text sequences.

**Notation:**
- $N$ = number of sequences in the dataset (e.g., $N = 1{,}000{,}000$ documents)
- $i$ = index over sequences, $i \in \{1, 2, \ldots, N\}$
- $T_i$ = length of sequence $i$ in tokens (each sequence can have different length)
- $x_t^{(i)}$ = the $t$-th token of the $i$-th sequence
- $\mathbf{x}^{(i)} = (x_1^{(i)}, x_2^{(i)}, \ldots, x_{T_i}^{(i)})$ = the entire $i$-th sequence
- $x_{<t}^{(i)} = (x_1^{(i)}, \ldots, x_{t-1}^{(i)})$ = all tokens before position $t$ in sequence $i$
- $\theta$ = all learnable parameters (weights) of the neural network

We want to find parameters $\theta$ such that our model assigns high probability to this data.

**The Likelihood Function:** The probability of observing one sequence under our model is:

$$P(\mathbf{x}^{(i)} | \theta) = P(x_1^{(i)}) \cdot P(x_2^{(i)} | x_1^{(i)}; \theta) \cdot P(x_3^{(i)} | x_1^{(i)}, x_2^{(i)}; \theta) \cdots$$

which we write more compactly as:

$$P(\mathbf{x}^{(i)} | \theta) = \prod_{t=1}^{T_i} P(x_t^{(i)} | x_{<t}^{(i)}; \theta)$$

> **Why the product of conditionals?** This is the chain rule of probability. For any sequence of events $A, B, C$:
> $$P(A, B, C) = P(A) \cdot P(B|A) \cdot P(C|A,B)$$
> 
> In words: the probability of seeing "the cat sat" equals:
> - $P(\text{the})$ — probability "the" starts the sequence
> - $\times\ P(\text{cat}|\text{the})$ — probability "cat" follows "the"
> - $\times\ P(\text{sat}|\text{the, cat})$ — probability "sat" follows "the cat"
>
> Each factor asks: "given everything so far, how likely is the next token?" This is exactly what our transformer computes at each position. The product gives us the probability of the entire sequence.

</br>**The Total Likelihood:** Assuming sequences are independent:

$$\mathcal{L}(\theta) = \prod_{i=1}^{N} P(\mathbf{x}^{(i)} | \theta) = \prod_{i=1}^{N} \prod_{t=1}^{T_i} P(x_t^{(i)} | x_{<t}^{(i)}; \theta)$$

The outer product is over all $N$ sequences; the inner product is over all $T_i$ token positions within sequence $i$.

> **What does the total likelihood mean?** We're asking: "Given our model with parameters $\theta$, what is the probability of observing exactly this dataset?" 
>
> The outer product $\prod_{i=1}^{N}$ multiplies across all sequences — we want to see sequence 1 AND sequence 2 AND ... AND sequence $N$. Since sequences are independent (one document doesn't affect another), we multiply their probabilities.
>
> The inner product $\prod_{t=1}^{T_i}$ handles one sequence — the probability of seeing token 1, then token 2, then token 3, etc., in that exact order.
>
> So $\mathcal{L}(\theta)$ answers: "How probable is it that we'd observe this exact training corpus — every sequence, every token, in exactly the order they appear — if text were generated by our model?" MLE finds the $\theta$ that makes this probability as large as possible, meaning the model "agrees" with the training data as much as it can.

</br>**Maximum Likelihood Estimation (MLE):** Find $\theta^*$ that maximizes $\mathcal{L}(\theta)$:

$$\theta^* = \arg\max_\theta \mathcal{L}(\theta)$$

**The Log-Likelihood:** Products are numerically unstable (multiplying many small probabilities → underflow) and hard to differentiate. Taking the logarithm converts products to sums:

$$\log \mathcal{L}(\theta) = \sum_{i=1}^{N} \sum_{t=1}^{T_i} \log P(x_t^{(i)} | x_{<t}^{(i)}; \theta)$$

Since $\log$ is monotonically increasing, $\arg\max_\theta \mathcal{L}(\theta) = \arg\max_\theta \log \mathcal{L}(\theta)$.

**Negative Log-Likelihood (NLL):** We prefer minimization in optimization (gradient descent minimizes), so:

$$\text{NLL}(\theta) = -\log \mathcal{L}(\theta) = -\sum_{i=1}^{N} \sum_{t=1}^{T_i} \log P(x_t^{(i)} | x_{<t}^{(i)}; \theta)$$

$$\theta^* = \arg\min_\theta \text{NLL}(\theta)$$

## 1.4 From NLL to Cross-Entropy Loss: The Derivation

Now let's connect this to the cross-entropy loss everyone uses. This is NOT a coincidence — it's the same thing.

**Step 1: The Model's Output**

For a given context $x_{<t}$, the model outputs a logit vector $\mathbf{z} \in \mathbb{R}^{|\mathcal{V}|}$:

$$\mathbf{z} = \begin{pmatrix} z_1 \\ z_2 \\ \vdots \\ z_{|\mathcal{V}|} \end{pmatrix}$$

These are raw scores, not probabilities. They can be any real numbers.

**Step 2: Softmax Converts Logits to Probabilities**

Softmax maps the logit vector to a probability vector $\hat{\mathbf{p}} \in \Delta^{|\mathcal{V}|-1}$:

$$\hat{\mathbf{p}} = \text{softmax}(\mathbf{z}) = \begin{pmatrix} \hat{p}_1 \\ \hat{p}_2 \\ \vdots \\ \hat{p}_{|\mathcal{V}|} \end{pmatrix} = \begin{pmatrix} \frac{e^{z_1}}{\sum_j e^{z_j}} \\ \frac{e^{z_2}}{\sum_j e^{z_j}} \\ \vdots \\ \frac{e^{z_{|\mathcal{V}|}}}{\sum_j e^{z_j}} \end{pmatrix}$$

**Why softmax?** It satisfies both constraints of a probability distribution:
- Non-negativity: $e^{z_k} > 0$ always, so $\hat{p}_k > 0$
- Normalization: By construction, $\sum_k \hat{p}_k = 1$

Moreover, softmax is differentiable, which we need for gradient descent.</br>

> **Note on Temperature:** Temperature $\tau$ is a scalar that scales the logits before softmax:
>
> $$\hat{p}_k = \frac{e^{z_k / \tau}}{\sum_j e^{z_j / \tau}}$$
>
> - $\tau = 1$: Standard softmax, no change.
> - $\tau < 1$ (e.g., 0.5): Dividing by a small number makes logits larger in magnitude. Differences between logits get amplified. The distribution becomes **sharper** — high probabilities get higher, low ones get lower. At $\tau \to 0$, it approaches a one-hot (argmax).
> - $\tau > 1$ (e.g., 2.0): Dividing by a large number shrinks logits toward zero. Differences get compressed. The distribution becomes **flatter**, closer to uniform. At $\tau \to \infty$, all tokens become equally likely.
>
> **Example:** Logits $[1.0, 2.0, 3.0]$
> - $\tau = 1.0$: softmax $\approx [0.09, 0.24, 0.67]$
> - $\tau = 0.5$: softmax $\approx [0.01, 0.12, 0.87]$ (sharper)
> - $\tau = 2.0$: softmax $\approx [0.19, 0.31, 0.50]$ (flatter)
>
> During training, we use $\tau = 1$. During inference, lower temperature gives more deterministic outputs (model picks its top choice), higher temperature gives more diverse/creative outputs (model samples more broadly).

</br>**Step 3: The Ground Truth as a One-Hot Vector**

The true next token is $x_t = k^*$ (some specific index). We represent this as a one-hot vector $\mathbf{p} \in \{0, 1\}^{|\mathcal{V}|}$:

$$\mathbf{p} = \mathbf{e}_{k^*} = \begin{pmatrix} 0 \\ \vdots \\ 0 \\ 1 \\ 0 \\ \vdots \\ 0 \end{pmatrix} \leftarrow \text{position } k^*$$

**Concrete example:** Vocabulary is $\{\text{the}, \text{cat}, \text{sat}, \text{mat}\}$ and the true token is "mat" ($k^* = 4$):

$$\mathbf{p} = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

This vector has exactly one 1 (at the true token's index) and zeros everywhere else. It represents a "degenerate" categorical distribution — 100% probability on the true token, 0% on everything else.

**Step 4: The NLL for One Prediction**

We want: what probability did our model assign to the true token?

That's simply the $k^*$-th element of the prediction vector: $\hat{p}_{k^*}$

We can extract this using a dot product with the one-hot vector:

$$\hat{p}_{k^*} = \mathbf{p}^\top \hat{\mathbf{p}} = \sum_{k=1}^{|\mathcal{V}|} p_k \hat{p}_k$$

**Verification with our example:** If $\mathbf{p} = (0, 0, 0, 1)^\top$ and $\hat{\mathbf{p}} = (0.028, 0.152, 0.068, 0.752)^\top$:

$$\mathbf{p}^\top \hat{\mathbf{p}} = 0 \cdot 0.028 + 0 \cdot 0.152 + 0 \cdot 0.068 + 1 \cdot 0.752 = 0.752 = \hat{p}_4$$

The dot product "selects" the probability at the true index.

The negative log-likelihood for this single prediction:

$$\text{NLL} = -\log \hat{p}_{k^*} = -\log(\mathbf{p}^\top \hat{\mathbf{p}})$$

**Step 5: Expressing This as Cross-Entropy**

Here's the key insight. Instead of $-\log(\mathbf{p}^\top \hat{\mathbf{p}})$, we can write:

$$-\log \hat{p}_{k^*} = -\sum_{k=1}^{|\mathcal{V}|} p_k \log \hat{p}_k = -\mathbf{p}^\top \log \hat{\mathbf{p}}$$

where $\log \hat{\mathbf{p}}$ means element-wise log:

$$\log \hat{\mathbf{p}} = \begin{pmatrix} \log \hat{p}_1 \\ \log \hat{p}_2 \\ \vdots \\ \log \hat{p}_{|\mathcal{V}|} \end{pmatrix}$$

**Why are these equivalent?** Because $\mathbf{p}$ is one-hot. Let's expand the sum:

$$-\sum_{k=1}^{|\mathcal{V}|} p_k \log \hat{p}_k = -(p_1 \log \hat{p}_1 + p_2 \log \hat{p}_2 + \cdots + p_{|\mathcal{V}|} \log \hat{p}_{|\mathcal{V}|})$$

Since $p_k = 0$ for all $k \neq k^*$ and $p_{k^*} = 1$:

$$= -(0 \cdot \log \hat{p}_1 + \cdots + 1 \cdot \log \hat{p}_{k^*} + \cdots + 0 \cdot \log \hat{p}_{|\mathcal{V}|}) = -\log \hat{p}_{k^*}$$

All terms vanish except the one at the true index.

**This is the cross-entropy formula:**

$$\boxed{H(\mathbf{p}, \hat{\mathbf{p}}) = -\sum_{k=1}^{|\mathcal{V}|} p_k \log \hat{p}_k = -\mathbf{p}^\top \log \hat{\mathbf{p}}}$$

> **Why bother with the sum notation?** In code you'd just write `loss = -log(probs[target])`. The sum form $-\sum_k p_k \log \hat{p}_k$ is useful because:
> 1. It generalizes to soft labels (when $\mathbf{p}$ is not one-hot, e.g., in knowledge distillation)
> 2. It connects to information theory (cross-entropy, KL divergence)
> 3. It makes derivations cleaner when proving properties of the loss

> **Conclusion: Loss Functions and Their Distributional Assumptions**
>
> We just derived cross-entropy loss directly from the categorical distribution. This is not a coincidence — every loss function implicitly assumes a probability distribution:
>
> | Loss Function | Assumed Distribution | What's Random? |
> |---------------|---------------------|----------------|
> | Mean Squared Error (MSE) | Gaussian | The residuals $\epsilon = y - \hat{y}$. Model predicts the mean, noise is additive: $y = f_\theta(x) + \epsilon$ where $\epsilon \sim \mathcal{N}(0, \sigma^2)$ |
> | Cross-Entropy | Categorical | The target itself. Model predicts the distribution directly: $\hat{\mathbf{p}} = f_\theta(x)$, no additive noise. |
> | Binary Cross-Entropy | Bernoulli | The target itself. Model predicts $P(y=1)$ directly. |
>
> **Key difference:** In regression (MSE), the noise is additive — we predict a value and reality adds noise to it. In classification (cross-entropy), there's no additive noise — we predict a probability distribution directly, and the randomness is inherent in sampling from that distribution.
>
> **Multi-class vs. Multi-label — these are NOT the same:**
>
> - **Multi-class (categorical):** Exactly one class is correct. Classes are mutually exclusive. Output is a single softmax over all classes, probabilities sum to 1. Example: "Is this image a cat, dog, or bird?" — it's exactly one of them.
>
> - **Multi-label:** Multiple classes can be correct simultaneously. Classes are independent. Output is a sigmoid per class (not softmax), each probability is independent, they don't sum to 1. Example: "What tags apply to this image?" — it can be "outdoor" AND "sunny" AND "beach" all at once.
>
> For language modeling, we use categorical cross-entropy because the next token is exactly one token from the vocabulary — it can't be half "cat" and half "dog". The softmax enforces this mutual exclusivity.

## 1.5 Cross-Entropy and Information Theory: The Deep Connection

Now we connect to information entropy. This matters because it reveals why reducing cross-entropy = reducing uncertainty = better predictions.

### 1.5.1 Shannon Entropy

**Definition (Entropy):** The entropy of a discrete random variable $X$ with PMF $\mathbf{p} = (p_1, p_2, \ldots, p_K)$ is:

$$H(X) = H(\mathbf{p}) = -\sum_{k=1}^{K} p_k \log p_k$$

with the convention that $0 \log 0 = 0$ (justified by continuity: $\lim_{x \to 0^+} x \log x = 0$).

**What does entropy measure?** Entropy quantifies the average "surprise" or uncertainty in a distribution. Think of it this way: if you're about to observe an outcome, how surprised will you be on average?

- If you already know what's going to happen (one outcome has probability 1), you won't be surprised at all → entropy is 0.
- If anything could happen with equal probability, you're maximally uncertain → entropy is maximal.

**The surprise of a single outcome:** Before defining average surprise, we need surprise for one outcome. If event $k$ has probability $p_k$, its surprise is:

$$\text{surprise}(k) = -\log p_k$$

Why this formula?
- Rare events ($p_k$ small) → $-\log p_k$ large → very surprising
- Common events ($p_k$ close to 1) → $-\log p_k$ close to 0 → not surprising
- Certain event ($p_k = 1$) → $-\log 1 = 0$ → zero surprise

**Entropy = expected surprise:** Entropy is just the average surprise, weighted by how often each outcome occurs:

$$H(\mathbf{p}) = \sum_{k=1}^{K} p_k \cdot \text{surprise}(k) = \sum_{k=1}^{K} p_k \cdot (-\log p_k) = -\sum_{k=1}^{K} p_k \log p_k$$

**Concrete Examples:**

**Example 1: Fair coin** — $\mathbf{p} = (0.5, 0.5)$

$$H(\mathbf{p}) = -[0.5 \log_2 0.5 + 0.5 \log_2 0.5] = -[0.5 \cdot (-1) + 0.5 \cdot (-1)] = 1 \text{ bit}$$

Interpretation: You need 1 bit of information to describe the outcome. Maximum uncertainty for 2 outcomes.

**Example 2: Biased coin** — $\mathbf{p} = (0.9, 0.1)$

$$H(\mathbf{p}) = -[0.9 \log_2 0.9 + 0.1 \log_2 0.1] = -[0.9 \cdot (-0.152) + 0.1 \cdot (-3.322)] = 0.137 + 0.332 = 0.469 \text{ bits}$$

Interpretation: Less uncertainty than a fair coin — you can mostly predict heads, so you're less surprised on average.

**Example 3: Certain outcome** — $\mathbf{p} = (1, 0)$

$$H(\mathbf{p}) = -[1 \cdot \log_2 1 + 0 \cdot \log_2 0] = -[1 \cdot 0 + 0] = 0 \text{ bits}$$

Interpretation: No uncertainty at all. You know exactly what will happen.

**Example 4: Uniform over 4 outcomes** — $\mathbf{p} = (0.25, 0.25, 0.25, 0.25)$

$$H(\mathbf{p}) = -4 \cdot [0.25 \log_2 0.25] = -4 \cdot [0.25 \cdot (-2)] = 2 \text{ bits}$$

Interpretation: Maximum uncertainty for 4 outcomes. You need 2 bits to specify which one occurred (like 2 coin flips: 00, 01, 10, 11).

**General rule for uniform distributions:** If $\mathbf{p}$ is uniform over $K$ outcomes, then $p_k = 1/K$ for all $k$, and:

$$H(\mathbf{p}) = -\sum_{k=1}^{K} \frac{1}{K} \log \frac{1}{K} = -K \cdot \frac{1}{K} \log \frac{1}{K} = -\log \frac{1}{K} = \log K$$

This is the maximum possible entropy for $K$ outcomes.

**Units:** 
- Using $\log_2$: entropy is in **bits** (binary digits needed to encode the outcome)
- Using $\ln$: entropy is in **nats** (natural units)
- Deep learning typically uses $\ln$ because it simplifies derivatives, but the interpretation is the same

**Connection to language modeling:** A language model that's very confident (puts 95% on one token) has low entropy predictions — it "knows" what comes next. A confused model (spreads probability across many tokens) has high entropy predictions — it's uncertain. Training reduces entropy by making the model more confident about the correct next token.

> **Note on Surprise, Logarithms, and Why They're Everywhere**
>
> **The core trick:** Surprise is inversely related to probability — the rarer an event, the more surprising it is when it occurs:
>
> $$\text{surprise}(k) = -\log p_k$$
>
> This makes intuitive sense: if your friend who never wins anything suddenly wins the lottery, you're shocked. If a professional athlete wins a race, you shrug.
>
> **Why logarithm specifically?** The logarithm is not arbitrary — it's the unique function that satisfies several desirable properties simultaneously:
>
> 1. **Inversion with compression:** Small probabilities must map to large surprises, but $1/p$ would explode (if $p = 0.0001$, then $1/p = 10000$). The logarithm inverts while compressing: $-\log(0.0001) \approx 9.2$ in nats. It keeps values manageable.
>
> 2. **Additivity:** Independent events should have additive surprises. If two independent events occur together, their combined probability is $p \cdot q$, and:
> $$-\log(p \cdot q) = -\log p + (-\log q)$$
> The logarithm is the *only* function that converts products to sums. This is crucial — it's why we can sum log-likelihoods across a dataset instead of multiplying tiny probabilities together.
>
> 3. **Numerical stability:** This is NOT a coincidence — it's the same reason we use log in loss functions. Consider a dataset of 1 million tokens. The likelihood is:
> $$\mathcal{L} = \prod_{i=1}^{1{,}000{,}000} p_i$$
> If each $p_i \approx 0.01$, then $\mathcal{L} \approx 0.01^{1{,}000{,}000} = 10^{-2{,}000{,}000}$. This is way below floating-point precision (~$10^{-308}$ for float64) — it underflows to exactly zero, and you lose all information.
>
> The log-likelihood fixes this:
> $$\log \mathcal{L} = \sum_{i=1}^{1{,}000{,}000} \log p_i \approx 1{,}000{,}000 \times (-4.6) = -4{,}600{,}000$$
> A perfectly reasonable number that fits in a float.
>
> **The deep connection:** Information theory (Shannon entropy), probability theory (MLE), and numerical computing all converge on the logarithm for the same fundamental reasons:
>
> | Domain | Problem | Log Solution |
> |--------|---------|--------------|
> | Information theory | Quantify surprise/uncertainty | $-\log p$ gives additive, bounded surprise |
> | Probability / MLE | Multiply many probabilities | $\sum \log p$ avoids underflow |
> | Optimization | Derivatives of products | $\frac{d}{d\theta} \log f(\theta) = \frac{f'(\theta)}{f(\theta)}$ — cleaner gradients |
> | Loss functions | Penalize confident wrong predictions | $-\log(0.01) = 4.6$ vs $-\log(0.99) = 0.01$ — strong gradient signal when wrong |
>
> **It's all the same log for the same reasons.** Shannon didn't invent a clever trick that happened to be useful for neural networks 60 years later — he identified a fundamental mathematical structure that appears whenever you're dealing with probabilities and information. The fact that it also saves us from numerical hell is a beautiful bonus (or perhaps an inevitable consequence of the same underlying math).

> **Note on Entropy and Real-World Storage**
>
> **Shannon's source coding theorem (1948):** Entropy is not just an abstract measure — it's the theoretical minimum number of bits required to encode data from a distribution. You physically cannot do better on average. This directly translates to RAM, disk space, and bandwidth.
>
> **Why low entropy = less storage:**
>
> Consider two files, both 1000 characters:
> - File A: "AAAAAAA..." (all A's) — You know every character is A. Instead of storing 8000 bits (1000 × 8 bits per ASCII char), you store "1000×A" — maybe 20 bits. Compression ratio: 400×
> - File B: Random bytes — Each byte is unpredictable. You need all 8000 bits. It won't compress at all.
>
> The difference? File A has near-zero entropy. File B has maximum entropy (8 bits per byte).
>
> **English text as a middle ground:**
>
> Raw ASCII uses 8 bits per character, but English has only ~1-1.5 bits of entropy per character. Why so low? Because language is predictable:
> - After "th", you can almost guarantee "e", "a", or "i"
> - After "q", you know "u" is coming
> - "the" appears far more often than "xzq"
>
> This redundancy is why text files compress to ~20% of their original size with gzip. The compression algorithm exploits the low entropy — it assigns short codes to common patterns ("the" → few bits) and long codes to rare ones ("xzq" → many bits).
>
> **Practical examples:**
>
> | Data type | Entropy per byte | Compresses? |
> |-----------|------------------|-------------|
> | File of zeros | ~0 bits | Yes — to almost nothing |
> | English text | ~1-2 bits | Yes — ~5× smaller |
> | Encrypted data | ~8 bits | No — looks random by design |
> | Already compressed (jpg, mp3) | ~8 bits | No — entropy already maximized |
>
> Try it yourself: run `gzip` on a text file versus on a .jpg. The text shrinks dramatically; the jpg barely changes.
>
> **Language models are compressors:**
>
> Here's the deep insight: a good language model is a good compressor, and vice versa. If your model predicts the next token with 95% confidence, you only need to encode "was it the top prediction or not?" — roughly $-\log_2(0.95) \approx 0.07$ bits if correct, more bits if wrong.
>
> This is arithmetic coding: use the model's predicted distribution to assign bit lengths. High-probability tokens get short codes, low-probability tokens get long codes. The expected bits per token equals the cross-entropy of the model.
>
> $$\text{Bits per token} = H(\mathbf{p}, \hat{\mathbf{p}}) = -\sum_k p_k \log_2 \hat{p}_k$$
>
> **Lower cross-entropy = better compression = less storage.**
>
> This is why researchers sometimes evaluate language models by compression ratio — it's mathematically equivalent to perplexity. A model with perplexity 10 needs $\log_2(10) \approx 3.3$ bits per token on average. A model with perplexity 100 needs $\log_2(100) \approx 6.6$ bits per token. The better model literally produces smaller files.
>
> **The profound implication:** Prediction and compression are two sides of the same coin. When OpenAI trains GPT on terabytes of text, they're essentially building the world's best text compressor. The model learns patterns, redundancies, and structure — everything that makes data predictable — because that's exactly what minimizes cross-entropy. Intelligence, in some sense, is compression.

### 1.5.2 Cross-Entropy Decomposition (not worked through)

**Definition (Cross-Entropy):** For two distributions $p$ (true) and $\hat{p}$ (predicted):

$$H(p, \hat{p}) = -\sum_{k=1}^{K} p_k \log \hat{p}_k$$

**The Key Decomposition:**

$$H(p, \hat{p}) = H(p) + D_{KL}(p \| \hat{p})$$

where $D_{KL}(p \| \hat{p})$ is the Kullback-Leibler divergence:

$$D_{KL}(p \| \hat{p}) = \sum_{k=1}^{K} p_k \log \frac{p_k}{\hat{p}_k}$$

**Proof:**

$$H(p, \hat{p}) = -\sum_k p_k \log \hat{p}_k$$

Add and subtract $\sum_k p_k \log p_k$:

$$= -\sum_k p_k \log p_k + \sum_k p_k \log p_k - \sum_k p_k \log \hat{p}_k$$

$$= H(p) + \sum_k p_k (\log p_k - \log \hat{p}_k)$$

$$= H(p) + \sum_k p_k \log \frac{p_k}{\hat{p}_k}$$

$$= H(p) + D_{KL}(p \| \hat{p})$$


### 1.5.3 Why "Cross" Entropy? (not worked through)

The name comes from the fact that we're measuring entropy "across" two distributions — using the true distribution $p$ to weight the log-probabilities from the predicted distribution $\hat{p}$.

If $\hat{p} = p$ (perfect prediction):

$$H(p, p) = -\sum_k p_k \log p_k = H(p)$$

Cross-entropy equals regular entropy when the prediction is perfect.

### 1.5.4 In Language Modeling: The One-Hot Case (not worked through)

For language modeling, $p$ is always one-hot (the true next token). One-hot distributions have zero entropy:

$$H(p_{\text{one-hot}}) = -(1 \cdot \log 1 + 0 \cdot \log 0 + \cdots) = 0$$

Therefore:

$$H(p_{\text{one-hot}}, \hat{p}) = 0 + D_{KL}(p_{\text{one-hot}} \| \hat{p}) = D_{KL}(p_{\text{one-hot}} \| \hat{p})$$

**Cross-entropy loss = KL divergence** when the true distribution is deterministic.

### 1.5.5 The Interpretation: Fighting Entropy (not worked through)

**Key Insight:** When we minimize cross-entropy, we're minimizing $H(p) + D_{KL}(p \| \hat{p})$. Since $H(p)$ is fixed (it's the true data distribution), we're really minimizing $D_{KL}(p \| \hat{p})$ — how different our prediction is from truth.

**The Entropy View:** The model starts with high entropy (uniform or near-uniform predictions — maximum uncertainty). Through training, it learns to concentrate probability mass on correct tokens, reducing the entropy of its predictions while maintaining the constraint that it must match the true conditional distribution.

**A perfect model** would have:
- $\hat{p} = p$ at every step
- $D_{KL}(p \| \hat{p}) = 0$
- Cross-entropy equals the irreducible entropy of language itself (which is non-zero because language has inherent unpredictability)

## 1.6 Likelihood vs. Probability: Training vs. Inference (not worked through)

Let's clear up a common confusion.

**During Training: We Compute Likelihoods**

The likelihood function $\mathcal{L}(\theta)$ treats the **data as fixed** and the **parameters as variable**:

$$\mathcal{L}(\theta) = P(\mathbf{x}^{(1)}, \ldots, \mathbf{x}^{(N)} | \theta)$$

We're asking: "Given this data, which parameters make it most probable?"

Technically, likelihood is a function of $\theta$, not a probability distribution over $\theta$. It doesn't integrate to 1. The notation $P(\mathbf{x} | \theta)$ can be read either way depending on what's fixed.

**During Inference: We Compute Probabilities**

At inference, parameters $\theta^*$ are fixed (we've trained the model). We compute:

$$P(x_t | x_{<t}; \theta^*) = \text{softmax}(f_{\theta^*}(x_{<t}))$$

This IS a proper probability distribution over the next token. It sums to 1.

**In Practice:** The distinction is subtle because the mathematical operations are identical. But conceptually:
- Training: Find $\theta$ that maximizes $P(\text{data} | \theta)$
- Inference: Use trained $\theta^*$ to compute $P(\text{next token} | \text{context}; \theta^*)$


## 1.7 Perplexity: The Exponential of Cross-Entropy (not worked through)

You'll often see language models evaluated by **perplexity** rather than cross-entropy.

**Definition:**

$$\text{Perplexity} = \exp\left(\frac{1}{T} \sum_{t=1}^T -\log P(x_t | x_{<t}; \theta)\right) = \exp(\text{average cross-entropy})$$

**Interpretation:** Perplexity represents the "effective vocabulary size" the model is choosing from. If perplexity = 100, it's as if the model is uniformly guessing among 100 tokens on average.

**Why use it?** 
- More interpretable than raw cross-entropy
- A perplexity of 1 would mean perfect prediction (only possible if language were deterministic)
- Human-level perplexity on English text is estimated around 15-25

---

# Part 2: The Transformer as an Entropy Reduction Machine

Now we understand the objective. The transformer's job is to minimize cross-entropy, which means making confident, accurate predictions. Every component exists to **extract information** from the input and reduce uncertainty in the output.

## 2.1 The Information-Theoretic View

Think of the raw input sequence as containing both **signal** (patterns that help prediction) and **noise** (irrelevant variation). The transformer progressively:

1. Extracts signal from the input
2. Combines information across positions
3. Refines representations layer by layer
4. Outputs a low-entropy distribution when confident, higher entropy when uncertain

**Each layer reduces conditional entropy of the prediction given the representation.** The final representation should capture everything needed for the prediction task.

## 2.2 Tokenization with BPE: Entropy-Optimal Encoding

### 2.2.1 Bits: The Fundamental Unit of Information

Before discussing tokens, we need to understand bits properly.

**One bit answers one yes/no question.** That's the definition. If I'm thinking of a number that's either 0 or 1, you need one question to find it: "Is it 1?" One bit.

**Two bits answer two yes/no questions.** If I'm thinking of a number in $\{0, 1, 2, 3\}$, you need two questions:
- "Is it ≥ 2?" (splits into {0,1} vs {2,3})
- "Is it odd?" (identifies the exact number)

Two bits distinguish 4 things. In binary: 00, 01, 10, 11.

**The pattern:** With $n$ bits, you can distinguish $2^n$ possibilities.

| Bits | Possibilities |
|------|---------------|
| 1    | $2^1 = 2$     |
| 2    | $2^2 = 4$     |
| 3    | $2^3 = 8$     |
| 8    | $2^8 = 256$   |
| 10   | $2^{10} = 1024$ |

**The inverse question:** If you have $K$ equally likely possibilities, how many bits do you need?

Solve $2^n = K$ for $n$:

$$n = \log_2 K$$

That's it. The logarithm answers: "How many times must I double 1 to reach $K$?" Or equivalently: "How many yes/no questions to identify one item among $K$?"

**Examples:**
- 256 ASCII characters: $\log_2(256) = 8$ bits per character
- 50,000 BPE tokens: $\log_2(50000) \approx 15.6$ bits per token
- 100,000 tokens: $\log_2(100000) \approx 16.6$ bits per token

This is the **maximum** information per token — achieved only when all tokens are equally likely. If some tokens are more probable, the actual information is lower (predictable things carry less information).

### 2.2.2 The Problem with Character-Level Models

Why not just use characters? Each character is one of ~256 bytes, so 8 bits max per position.

> **Note: Why 8 bits per character?**
>
> A **bit** is the smallest unit of information — it has two states: 0 or 1. 
>
> A **byte** is 8 bits grouped together. Why 8? Historical convention from early computing, but it stuck because $2^8 = 256$ is a convenient number — enough for all English letters (upper and lower), digits, punctuation, and control characters.
>
> With 8 bits, you can form $2^8 = 256$ unique patterns:
> ```
> 00000000 = 0
> 00000001 = 1
> 00000010 = 2
> ...
> 11111111 = 255
> ```
>
> ASCII encoding assigns each pattern to a character:
> ```
> 01000001 = 65 = 'A'
> 01000010 = 66 = 'B'
> 01100001 = 97 = 'a'
> 00110000 = 48 = '0'
> ```
>
> So when we say "8 bits per character," we're saying: to specify one of 256 possible characters, you must answer 8 binary questions. That's $\log_2(256) = 8$.
>
> This is the **maximum** information per character — achieved only if all 256 characters were equally likely. In practice, 'e' appears far more often than 'z', so the actual entropy of English text is much lower (~1.5 bits per character). But the encoding still uses 8 bits regardless — that's the inefficiency that compression algorithms exploit.

**Problem:** Consider the word "information" — 11 characters. The model processes 11 positions to handle one concept. Each position carries at most 8 bits.

With a 50,000-token vocabulary, "information" might be a single token. One position, up to 15.6 bits of potential information.

**The tradeoff:**
- Characters: Small vocabulary (256), many positions, low bits per position
- BPE: Large vocabulary (50k), fewer positions, high bits per position

Transformers are expensive per position (attention is $O(T^2)$). Packing more information per position is efficient.

### 2.2.3 What BPE Does

**Byte Pair Encoding (BPE)** iteratively merges the most frequent adjacent pairs:

1. Start with character vocabulary
2. Count all adjacent pairs in corpus
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary reaches target size

**Example:**
```
Corpus: "low lower lowest low"
Initial: ['l', 'o', 'w', ' ', 'l', 'o', 'w', 'e', 'r', ...]
After merge 'lo': ['lo', 'w', ' ', 'lo', 'w', 'e', 'r', ...]
After merge 'low': ['low', ' ', 'low', 'e', 'r', ...]
...
```

### 2.2.4 Why BPE is Information-Theoretically Motivated

Recall Shannon's source coding theorem: you need at least $H(X)$ bits on average to encode data from distribution $X$. You achieve this by giving short codes to frequent symbols, long codes to rare ones.

**Huffman coding** does this explicitly: "the" (frequent) gets a short bit string, "xylophone" (rare) gets a long one.

**BPE does something complementary:** Instead of varying code length, it varies what counts as a symbol. Frequent sequences become single tokens. The result:
- "the" = 1 token (frequent, so it earned its own symbol)
- "xylophone" = maybe 3 tokens ["xy", "lo", "phone"] (rare, stays decomposed)

**The effect:** Token frequencies become more uniform. When frequencies are uniform, each token carries close to $\log_2 |\mathcal{V}|$ bits — we're using our vocabulary efficiently.

### 2.2.5 Information Per Token in Practice

A vocabulary of 50,000 gives a maximum of $\log_2(50000) \approx 15.6$ bits per token position.

But actual information depends on context:
- "I went to the ___" → "store" is predictable → low information (maybe 2-3 bits)
- "My name is ___" → could be anything → high information (maybe 12+ bits)

The model's job: figure out how much information each position actually carries, given context. When it's confident, it outputs a low-entropy distribution (few bits of uncertainty remain). When it's uncertain, high entropy (many bits still unknown).

**The connection to loss:** Cross-entropy measures exactly this — how many bits of surprise remain after seeing the model's prediction. Lower cross-entropy = fewer bits = better compression = better prediction.